# Three-field multiscale Darcy: MH²M

Two independent interface spaces implement equations (25)–(29) of [de Barros, Madureira and Valentin, arXiv:2404.16978v3](https://arxiv.org/abs/2404.16978v3). The global unknown is the continuous pressure trace Γ. The local conormal Λ uses λ=A∇p·n, the negative of the physical outward Darcy flux.

The current main equations and their physical checks execute from source. The full acquisition and its field galleries run with `--study`; inspect `--study --plan` for locked commands and resource requirements. Exact original external comparisons require `--historical` and their attributed payloads.


The PyMHM distribution contains only the library. The first cell explicitly
downloads a checksum-verified companion archive and prepares the declared data
using its local Python helpers. You can inspect these support files in `ROOT`.
Complete studies and historical replay retain their existing opt-in flags.


In [ ]:
from pathlib import Path
import os
import sys
from pymhm.io.workspace import workspace_from_archive

# Download verified support files; this operation does not execute them.
COMPANION_URL = "https://ipes-lncc.github.io/pymhm/downloads/a53fe2c45a6c3da73d06d8cfd6297e22e92bc7b26dafd13bac60794be59c1750/52_mh2m-companion.zip"
COMPANION_SHA256 = "a53fe2c45a6c3da73d06d8cfd6297e22e92bc7b26dafd13bac60794be59c1750"
WORKSPACE = Path(
    os.environ.get("PYMHM_WORKSPACE", Path.cwd() / ".pymhm-companions" / COMPANION_SHA256)
)
ROOT = workspace_from_archive(COMPANION_URL, sha256=COMPANION_SHA256, directory=WORKSPACE)
os.environ["PYMHM_WORKSPACE"] = str(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Explicitly prepare declared data with the downloaded Python helpers.
from scripts.notebook_reproduction import notebook_workspace

ROOT = notebook_workspace("darcy/52_mh2m.ipynb", directory=ROOT)
root = ROOT
print("Workspace:", ROOT)


import os

HISTORICAL_REPLAY = os.environ.get("PYMHM_NOTEBOOK_HISTORICAL", "0") == "1"
STUDY_REPLAY = os.environ.get("PYMHM_NOTEBOOK_STUDY", "0") == "1"


In [ ]:
from pathlib import Path
import json
import numpy as np
from IPython.display import Image, display
from threadpoolctl import threadpool_limits
from pymhm.meshes.triangle import TriangleMesh
from pymhm.fem.traces.interval import SkeletonSpace, FaceSpace
from pymhm.fem.traces.pressure_2d import PressureTraceSpace


## Small analytical verification

A nonharmonic quadratic with nonzero Dirichlet data checks the complete source reconstruction. The independent campaign below uses the quartic pressure from Section 8.1.

## Declare the three-field equations

The application declares the coupled local pressure/conormal saddle and its
global continuous pressure-trace test, using the independently chosen spaces:

$$
\begin{aligned}
A_T&=\begin{bmatrix}K_T&-\mathcal{B}_T\\-\mathcal{B}_T^{\mathsf T}&0\end{bmatrix},
&B_T&=\begin{bmatrix}0\\P_T\end{bmatrix},\\
C_T&=B_T^{\mathsf T}, &D_T&=0, &f_T&=\begin{bmatrix}F_T\\0\end{bmatrix}.
\end{aligned}
$$

Local coefficients are pressure and outward conormal eta=K grad(p).n; its
negative is physical Darcy normal flux. Gamma shares pressure nodes, while
Lambda remains private to each macrocell. The global `Equation` supplies
outward Neumann moments and fixed Gamma pressure values. Pure Neumann data add
the physical integral of the complete reconstructed pressure. Generic assembly
and solve consume these blocks directly. The shared Neumann-map kernel retains
the original boundary-mean basis for inspection and injectivity checks.


In [ ]:
from pymhm import assemble
from examples.formulations.darcy import pressure_constraints
from examples.formulations.three_field import define_three_field, recover_three_field


def exact(x):
    """Quadratic pressure with nonzero boundary values."""
    return x[:, 0] ** 2 + x[:, 0] * x[:, 1] + 2 * x[:, 1] ** 2


mesh = TriangleMesh.unit_square(2)
with threadpool_limits(1):
    solution_definition = define_three_field(
        mesh,
        source=-6,
        dirichlet=exact,
        degree=2,
        local_refinement=2,
        pressure_trace=PressureTraceSpace.uniform(mesh, 2),
        flux_space=SkeletonSpace(mesh, tuple((FaceSpace.uniform(1) for _ in mesh.faces))),
    )
    solution_system = assemble(solution_definition.problem)
    solution = recover_three_field(
        solution_definition,
        solution_system,
        solution_system.solve(
            constraints=pressure_constraints(solution_definition, solution_system)
        ),
    )
assert solution.l2_error(exact) < 1e-11
assert np.max(abs(solution.conservation_residuals())) < 1e-11
{"pressure_error_l2": solution.l2_error(exact), "global_residual": solution.residual}


## Smooth published data and independent space refinements

The full study contains three polynomial families and five macro resolutions. The oscillatory coefficient uses equation (61), with explicitly selected γ=1 and the smooth source. The article does not identify these historical data completely; this is a declared-data experiment, not equality with its plotted fields. The numerical P1 reference has its own refinement study.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    record = json.loads((ROOT / "examples/results/mh2m/comparison.json").read_text())
    pressure = json.loads((ROOT / "examples/results/mh2m/pressure-enrichment.json").read_text())
    assert record["source_changed_during_run"] is False
    assert pressure["source_changed_during_run"] is False
    [(row["flux_segments"], row["free_trace_dofs"], row["flux_relative_difference"])
     for row in record["enrichment"]]

Conormal enrichment leaves nine free global unknowns and does not reduce the error of the unresolved fixed pressure trace. Refining Γ separately reduces the flux difference from 38.05% to 4.334%; the last classical-reference refinement changes the flux by 4.170%. A small algebraic residual does not remove this approximation/reference uncertainty.

In [ ]:
if STUDY_REPLAY or HISTORICAL_REPLAY:
    for name in ["convergence", "smooth-fields", "enrichment", "pressure-trace-enrichment",
                 "oscillatory-pressure", "oscillatory-flux"]:
        display(Image(filename=str(ROOT / "docs/figures/mh2m" / f"{name}.png")))

Reacquire with `python -m examples.mh2m_campaign` and `python -m examples.mh2m_trace_campaign`. Render archived data with `python -m examples.plot_mh2m`. The source notebook executes only the small quadratic check; numerical campaigns remain separate from light CI.